# ☕ Coffee Shop EDA — Optimized Visualization Strategy

This notebook explores coffee shop transaction data by selecting the absolute best library (**Matplotlib**, **Seaborn**, or **Plotly**) for each specific business question, rather than overlapping them in a redundant fashion. The visual theme uses a clean **teal and blue palette**.

### The Curated Library Selection

| Chart Type | Library Chosen | Rationale |
| :--- | :--- | :--- |
| **Bar / Horizontal Bar** | **Matplotlib** | Direct, precise control over layout and rankings with low overhead. |
| **Count Plot / Histogram / Heatmap** | **Seaborn** | Handles rapid statistical slicing, automatic counting, annotations, and distribution lines smoothly. |
| **Pie / Line / Box Plot** | **Plotly** | Unmatched value for interactivity: zoomable time series, exact hover data for distributions, and crisp slice percentages. |

---
## Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Apply a clean aesthetic baseline
sns.set_style('whitegrid')
print('✅ Optimal environment initialized!')

---
## Load & Inspect the Data

In [ ]:
df = pd.read_csv(
    'coffee_shop_transactions_cleaned.csv',
    parse_dates=['DateTime']
)

# Extract time dimensions for downstream patterns
df['Date'] = df['DateTime'].dt.date
df['Hour'] = df['DateTime'].dt.hour
df['DayOfWeek'] = df['DateTime'].dt.day_name()

print(f"Shape: {df.shape[0]} transactions, {df.shape[1]} columns")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

---
## 1️⃣ Bar Chart — Compare Categories (Matplotlib)
**Question:** Which items generate the most total revenue?  
*Why Matplotlib? Excellent for setting explicit categorical constraints and sizing.*

In [ ]:
revenue_by_item = df.groupby('Item')['TotalPrice'].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(revenue_by_item.index, revenue_by_item.values, color='#008080') # Teal
plt.xlabel('Item', fontsize=11, fontweight='bold')
plt.ylabel('Total Revenue ($)', fontsize=11, fontweight='bold')
plt.title('Revenue Contribution by Item', fontsize=13, fontweight='bold', pad=15)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## 2️⃣ Horizontal Bar Chart — Rank Categories (Matplotlib)
**Question:** Which items sold the most units?  
*Why Matplotlib? Simple orientation shifts make ranked item reading highly intuitive.*

In [ ]:
qty_by_item = df.groupby('Item')['Quantity'].sum().sort_values()

plt.figure(figsize=(8, 5))
plt.barh(qty_by_item.index, qty_by_item.values, color='#4682B4') # Steel Blue
plt.xlabel('Units Sold', fontsize=11, fontweight='bold')
plt.ylabel('Item', fontsize=11, fontweight='bold')
plt.title('Product Performance Rankings (Units Sold)', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---
## 3️⃣ Pie Chart — Show Percentages of a Whole (Plotly)
**Question:** What share of total revenue comes from each payment method?  
*Why Plotly? Hover mechanics dynamically show raw volume alongside exact percentages.*

In [ ]:
revenue_by_payment = df.groupby('PaymentMethod')['TotalPrice'].sum()

fig = px.pie(
    values=revenue_by_payment.values, 
    names=revenue_by_payment.index,
    title='Revenue Split by Payment Method',
    color_discrete_sequence=px.colors.sequential.Blues_r # Dark Blue to Light Blue
)
fig.update_traces(textinfo='percent+label', hole=0.1)
fig.update_layout(title_font=dict(size=16, family="Arial", color="black"))
fig.show()

---
## 4️⃣ Line Chart — Show Trends Over Time (Plotly)
**Question:** How does daily revenue trend across the month?  
*Why Plotly? Essential for time-series timelines where you need to isolate or hover over specific outlier dates.*

In [ ]:
revenue_by_date = df.groupby('Date')['TotalPrice'].sum().reset_index()

fig = px.line(
    revenue_by_date, x='Date', y='TotalPrice', markers=True,
    labels={'TotalPrice': 'Daily Revenue ($)'},
    title='Daily Financial Revenue Trends'
)
fig.update_traces(line_color='#005b96', marker=dict(size=6, color='#002b49')) # Deep Blue Theme
fig.update_layout(title_font=dict(size=16))
fig.show()

In [ ]:
# 1. Prepare your time-series data
revenue_by_date = df.groupby('Date')['TotalPrice'].sum().reset_index()

# 2. Build the standard Plotly line chart
fig = px.line(
    revenue_by_date, x='Date', y='TotalPrice', markers=True,
    text='TotalPrice', # Maps value labels onto data points
    labels={'TotalPrice': 'Daily Revenue ($)'},
    title='Daily Financial Revenue Trends (With Zoom & Sliders)'
)

# 3. Apply your custom styling and theme colors
fig.update_traces(
    line_color='#005b96',                  # Deep Blue Line
    marker=dict(size=6, color='#002b49'),  # Dark Blue Markers
    textposition="top center",
    texttemplate='$%{text:.2f}'            # Label format with 2 decimals
)

# 4. CRUCIAL: Inject the Range Slider and Selection Presets
fig.update_xaxes(
    rangeslider_visible=True,              # Activates the bottom timeline slider
    rangeselector=dict(
        buttons=list([
            dict(count=7, label="1w", step="day", stepmode="backward"),   # View last 7 days
            dict(count=1, label="1m", step="month", stepmode="backward"), # View last 1 month
            dict(step="all", label="All")                                 # Reset view
        ]),
        font=dict(size=11, color="#002b49"),
        bgcolor="#e0f2f1",                 # Soft teal background for buttons
        activecolor="#008080"              # Vibrant teal accent for the clicked button
    )
)

# 5. Fine-tune layout geometry
fig.update_layout(
    title_font=dict(size=16, family="Arial"),
    xaxis_title="Transaction Date",
    yaxis_title="Total Collected Revenue ($)",
    height=600 # Expanded height to make room for the bottom slider
)

fig.show()

---
## 5️⃣ Count Plot — Count Occurrences (Seaborn)
**Question:** How many transactions were processed per payment stream?  
*Why Seaborn? `countplot` executes aggregations directly from the source rows natively without manual groupings.*

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(
    data=df, x='PaymentMethod', hue='PaymentMethod',
    palette='crest', legend=False, # Teal / Green-Blue gradient
    order=df['PaymentMethod'].value_counts().index
)
plt.xlabel('Payment Method', fontsize=11, fontweight='bold')
plt.ylabel('Number of Transactions', fontsize=11, fontweight='bold')
plt.title('Transaction Count by Channel', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---
## 6️⃣ Histogram — Show Distributions (Seaborn)
**Question:** What does the distribution of transaction amounts look like?  
*Why Seaborn? Integrates Kernel Density Estimation (KDE) line to map the probability curve seamlessly.*

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['TotalPrice'], bins=8, color='#008080', kde=True, edgecolor='white') # Teal
plt.xlabel('Total Order Price ($)', fontsize=11, fontweight='bold')
plt.ylabel('Frequency (Log Scale Context)', fontsize=11, fontweight='bold')
plt.title('Distribution of Customer Order Sizes', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---
## 7️⃣ Box Plot — Show Spread and Outliers (Plotly)
**Question:** How does the transaction price vary across items?  
*Why Plotly? Instantly provides the min, maximum, median, and exact inner-quartile ranges upon hover.*

In [ ]:
fig = px.box(
    df, x='Item', y='TotalPrice',
    title='Price Distribution & Ticket Dispersion by Item',
    color='Item', 
    color_discrete_sequence=px.colors.sequential.Teal
)
fig.update_layout(showlegend=False, title_font=dict(size=16), yaxis_title="Total Price ($)")
fig.show()

---
## 8️⃣ Heatmap — Show Relationships (Seaborn)
**Question:** How are PricePerItem, Quantity, TotalPrice, and Hour correlated?  
*Why Seaborn? Highly optimized matrix plotting that applies data annotations directly into clean color maps with minimal code.*

In [ ]:
corr_matrix = df[['PricePerItem', 'Quantity', 'TotalPrice', 'Hour']].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(
    corr_matrix, annot=True, cmap='YlGnBu', vmin=-1, vmax=1, fmt='.2f',
    cbar_kws={'label': 'Correlation Coefficient'}
)
plt.title('Feature Correlation Heatmap Matrix', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---
## EDA Summary & Conclusions

| Metric Target | Chart Typology | Selected Library | Strategic Data Finding |
| :--- | :--- | :--- | :--- |
| **Sales Performance** | Vertical Bar | **Matplotlib** | Identifies absolute monetary leaders across item catalogs. |
| **Volume Velocity** | Horizontal Bar | **Matplotlib** | Pinpoints exact operational inventory pull speed. |
| **Mix Percentages** | Pie Donut | **Plotly** | Highlights consumer transaction channel preference shares. |
| **Timeline Variance**| Interactive Line | **Plotly** | Maps macro temporal performance, drops, or spikes over weeks. |
| **Traffic Counts** | Count Plot | **Seaborn** | Maps raw footprint traffic patterns without bias. |
| **Pricing Slices** | Histogram Distribution | **Seaborn** | Highlights common order ranges and purchase frequencies. |
| **Outlier Variance** | Statistical Box Plot | **Plotly** | Exposes pricing variance ranges and anomalies across sub-products. |
| **Variable Synergy** | Covariance Heatmap | **Seaborn** | Maps linear relationships between operational hours and tickets. |